# Handling Skewed Features

**Topic:** Feature Engineering

*This notebook works with `price`, the column we are trying to predict, because it is the most skewed column in this dataset and the effect of transforming it is dramatic and easy to see. The same four transforms below (log, square root, Box-Cox, Yeo-Johnson) apply just as directly to a skewed input feature — the difference is that a target has to be converted back afterward, and a feature does not.*

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import Dropdown, Output, VBox
from IPython.display import display, clear_output
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PowerTransformer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from tkh_utils import PALETTE, FONT, base_layout, load_airbnb


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** what a right-skewed distribution looks like and why a few extreme values pull the mean away from the typical case
- **Explain** why you can't compare R² across a raw-target model and a transformed-target model directly, and how scoring both back on the original dollar scale fixes that
- **Interpret** a side-by-side comparison of R², MAE, and median dollar error to judge whether a transform actually helped

> **Tip:** Start with `log1p` and watch the R² number and the dollar numbers below it. They don't move in the same direction — that's the whole lesson.

---
## How we got here

In **[05_polynomial_features.ipynb](05_polynomial_features.ipynb)** you saw a feature's relationship with price change shape depending on the scale it's measured on. In **[statistics/06_shape_of_distributions.ipynb](../statistics/06_shape_of_distributions.ipynb)** you learned what skewness measures and how to read it.

Now those two threads meet: a skewed target (or feature) is one where the raw scale is working against you, not the underlying relationship. Transforming it fixes the scale before modeling begins — but as you're about to see, "fixing" the scale is not free.

---
## Why this matters for data science

`price` in this dataset is right-skewed: its skewness is 3.03. Most listings sit under $150 a night, but a small number run past $1,000, and those few expensive listings drag the mean well above what a typical listing actually costs.

A log transform — and its relatives, square root, Box-Cox, and Yeo-Johnson — compresses that tail. After `log1p`, the skewness of `price` drops to about -0.02, which is close to perfectly symmetric.

That sounds like a strict improvement. It isn't. Compressing the tail means the model stops trying as hard to get the $1,000+ listings exactly right, because on the log scale being off by $500 on an expensive listing barely registers. That trade-off only becomes visible once you score both models on the same scale: R² can go down while the typical listing's prediction gets noticeably more accurate. Which one is "better" depends on whether you care more about the rare expensive listing or the typical cheap one — the widget below shows you both numbers at once, so you don't have to guess.

---
## Try it yourself

In [ ]:
_airbnb = load_airbnb()
_airbnb = _airbnb[_airbnb['price'] > 0].copy()

_TIMES_SQUARE_LAT, _TIMES_SQUARE_LON = 40.7580, -73.9855
_airbnb['dist_midtown'] = np.sqrt(
    (_airbnb['latitude'] - _TIMES_SQUARE_LAT) ** 2 +
    (_airbnb['longitude'] - _TIMES_SQUARE_LON) ** 2
)

_feat_cols = ['room_type', 'neighbourhood_group', 'dist_midtown']
_Xtr, _Xva, _ytr, _yva = train_test_split(
    _airbnb[_feat_cols], _airbnb['price'], test_size=0.25, random_state=42)

_prep = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['room_type', 'neighbourhood_group']),
    ('num', StandardScaler(), ['dist_midtown']),
])

def _forward(vals, method, fitted=None):
    """Apply a transform. For Box-Cox/Yeo-Johnson, fit only if `fitted` is None —
    this is what keeps the validation split from leaking into the transformer."""
    vals = np.asarray(vals, dtype=float)
    if method == "None":
        return vals, None
    if method == "log1p":
        return np.log1p(vals), None
    if method == "sqrt":
        return np.sqrt(vals), None
    power_method = "box-cox" if method == "Box-Cox" else "yeo-johnson"
    if fitted is None:
        fitted = PowerTransformer(method=power_method)
        return fitted.fit_transform(vals.reshape(-1, 1)).ravel(), fitted
    return fitted.transform(vals.reshape(-1, 1)).ravel(), fitted

def _inverse(vals, method, fitted=None):
    vals = np.asarray(vals, dtype=float)
    if method == "None":
        return vals
    if method == "log1p":
        return np.expm1(vals)
    if method == "sqrt":
        return vals ** 2
    return fitted.inverse_transform(vals.reshape(-1, 1)).ravel()

out_skew = Output()
caption_skew = widgets.HTML()

transform_dd = Dropdown(
    options=["None", "log1p", "sqrt", "Box-Cox", "Yeo-Johnson"], value="log1p",
    description="Transform:", style={"description_width": "80px"},
    layout=widgets.Layout(width="300px"))

_initialized = False

def render_skew(change=None):
    method = transform_dd.value

    # Display-only transform of the full column, for the two histograms.
    full_transformed, _ = _forward(_airbnb['price'].values, method)
    raw_skew = float(pd.Series(_airbnb['price'].values).skew())
    trans_skew = float(pd.Series(full_transformed).skew())

    # Raw-target model: trained and scored in dollars.
    raw_pipe = Pipeline([('prep', _prep), ('lr', LinearRegression())])
    raw_pipe.fit(_Xtr, _ytr)
    raw_pred = raw_pipe.predict(_Xva)
    r2_raw = r2_score(_yva, raw_pred)
    mae_raw = mean_absolute_error(_yva, raw_pred)
    med_raw = float(np.median(np.abs(_yva.values - raw_pred)))

    # Transformed-target model: fit the transformer on train only, then use it
    # (never refit it) to convert predictions back to dollars before scoring.
    ytr_t, fitted_transformer = _forward(_ytr.values, method)
    trans_pipe = Pipeline([('prep', _prep), ('lr', LinearRegression())])
    trans_pipe.fit(_Xtr, ytr_t)
    pred_t = trans_pipe.predict(_Xva)
    pred_dollars = _inverse(pred_t, method, fitted_transformer)
    r2_trans = r2_score(_yva, pred_dollars)
    mae_trans = mean_absolute_error(_yva, pred_dollars)
    med_trans = float(np.median(np.abs(_yva.values - pred_dollars)))

    fig = make_subplots(
        rows=1, cols=3,
        specs=[[{}, {}, {"secondary_y": True}]],
        subplot_titles=(
            f"Raw price (skew={raw_skew:.2f})",
            f"{method} (skew={trans_skew:.2f})",
            "Accuracy, scored in dollars",
        ),
    )
    fig.add_trace(go.Histogram(
        x=_airbnb['price'], nbinsx=50, marker_color=PALETTE["muted"]), row=1, col=1)
    fig.add_trace(go.Histogram(
        x=full_transformed, nbinsx=50, marker_color=PALETTE["primary"]), row=1, col=2)

    fig.add_trace(go.Bar(
        x=["R²"], y=[r2_raw], name="Raw price model", marker_color=PALETTE["muted"],
        text=[f"{r2_raw:.3f}"], textposition="outside", legendgroup="raw",
    ), row=1, col=3, secondary_y=False)
    fig.add_trace(go.Bar(
        x=["R²"], y=[r2_trans], name=f"{method} model", marker_color=PALETTE["primary"],
        text=[f"{r2_trans:.3f}"], textposition="outside", legendgroup="trans",
    ), row=1, col=3, secondary_y=False)
    fig.add_trace(go.Bar(
        x=["MAE ($)", "Median error ($)"], y=[mae_raw, med_raw], name="Raw price model",
        marker_color=PALETTE["muted"], text=[f"${mae_raw:.0f}", f"${med_raw:.0f}"],
        textposition="outside", legendgroup="raw", showlegend=False,
    ), row=1, col=3, secondary_y=True)
    fig.add_trace(go.Bar(
        x=["MAE ($)", "Median error ($)"], y=[mae_trans, med_trans], name=f"{method} model",
        marker_color=PALETTE["primary"], text=[f"${mae_trans:.0f}", f"${med_trans:.0f}"],
        textposition="outside", legendgroup="trans", showlegend=False,
    ), row=1, col=3, secondary_y=True)

    fig.update_layout(
        height=440, showlegend=True, barmode="group",
        paper_bgcolor=PALETTE["background"], plot_bgcolor=PALETTE["surface"],
        font=dict(family=FONT["family"], size=FONT["size_tick"]),
    )
    fig.update_xaxes(gridcolor="#DEE2E6", title_font=dict(size=FONT["size_axis"]))
    fig.update_yaxes(gridcolor="#DEE2E6", title_font=dict(size=FONT["size_axis"]))
    fig.update_xaxes(title_text="Price ($)", row=1, col=1)
    fig.update_xaxes(title_text=(f"{method}(price)" if method != "None" else "price"), row=1, col=2)
    fig.update_yaxes(title_text="Count", row=1, col=1)
    fig.update_yaxes(title_text="Count", row=1, col=2)
    fig.update_yaxes(title_text="R² (higher is better)", row=1, col=3, secondary_y=False, range=[0, 0.4])
    fig.update_yaxes(title_text="Dollars (lower is better)", row=1, col=3, secondary_y=True)

    with out_skew:
        clear_output(wait=True)
        fig.show()

    r2_delta = r2_trans - r2_raw
    med_delta = med_trans - med_raw
    r2_move = "stayed at" if abs(r2_delta) < 1e-9 else ("fell from" if r2_delta < 0 else "rose from")
    med_move = "stayed at" if abs(med_delta) < 1e-9 else ("dropped from" if med_delta < 0 else "rose from")
    r2_text = (f"R² {r2_move} {r2_raw:.3f}" if r2_move == "stayed at"
               else f"R² {r2_move} {r2_raw:.3f} to {r2_trans:.3f}")
    med_text = (f"the typical listing's error {med_move} ${med_raw:.0f}" if med_move == "stayed at"
                else f"the typical listing's error {med_move} ${med_raw:.0f} to ${med_trans:.0f}")
    caption_skew.value = (
        f"<b>{method}:</b> price skew {raw_skew:.2f} \u2192 {trans_skew:.2f}. "
        f"{r2_text}, while {med_text}."
    )

transform_dd.observe(render_skew, names="value")
display(VBox([transform_dd, out_skew, caption_skew]))
render_skew()


---
## What's happening?

Skewness measures how lopsided a distribution is. Zero is symmetric; a positive number means a right tail — a handful of large values pulling the mean up past where most of the data sits. Price's raw skew of 3.03 means most listings cluster under $150, with a thin scatter running past $1,000.

The four transforms above compress that tail by different amounts, from gentlest to strongest: square root, log1p, Box-Cox, Yeo-Johnson. All four shrink large values proportionally more than small ones, pulling the distribution back toward the center.

**Why R² alone can't tell you which model is "better":** R² measures the fraction of the target's own variance a model explains. Change the target from dollars to log-dollars and you change what "variance" means, so the two R² numbers are not measuring the same thing — comparing them directly is comparing scores from two different tests. The fix used above: fit the model on the transformed target, then convert every prediction back into dollars with the inverse transform (`np.expm1`, squaring, or the fitted `PowerTransformer`'s own `inverse_transform`) before scoring. Both models are then graded on the same test — dollars.

**A leakage trap worth knowing about:** Box-Cox and Yeo-Johnson each fit a transformer — they search for the optimal power to apply. That transformer must be fit on the training target only, then applied (never refit) wherever else it's needed. Fitting it again on validation data means the model is being scored against a slightly different scale than it was trained on. That's the same category of mistake covered in **[preprocessing/08_data_leakage.ipynb](../preprocessing/08_data_leakage.ipynb)**: information the model shouldn't have yet leaking into how it gets evaluated.

### Choosing a transform

| Transformation | Formula | Best for | Handles zero/negative? | sklearn class |
|----------------|---------|---------|----------------------|---------------|
| log1p | log(x + 1) | Prices, counts | Zero: yes. Negative: no | `FunctionTransformer(np.log1p)` |
| Square root | √x | Mild right skew | Zero: yes. Negative: no | `FunctionTransformer(np.sqrt)` |
| Box-Cox | optimal power | Strong right skew | No | `PowerTransformer(method='box-cox')` |
| Yeo-Johnson | optimal power | Any skew | Yes | `PowerTransformer(method='yeo-johnson')` |

### Measured on this dataset

Features: `room_type` + `neighbourhood_group` (one-hot) + `dist_midtown`. Trained on the transformed target, every prediction converted back to dollars before scoring.

| Target | Skew before → after | R² (dollars) | MAE | Median error |
|---|---|---|---|---|
| Raw price | 3.03 → 3.03 | 0.286 | $78.95 | $55.98 |
| log1p(price) | 3.03 → -0.02 | 0.248 | $74.62 | $44.68 |
| sqrt(price) | 3.03 → 1.14 | 0.278 | $75.45 | $48.53 |
| Box-Cox(price) | 3.03 → 0.00 | 0.249 | $74.61 | $44.73 |
| Yeo-Johnson(price) | 3.03 → 0.00 | 0.249 | $74.61 | $44.73 |

Every transform lowers R² and lowers the typical listing's error at the same time. Square root, the gentlest of the four, lands in between on every column — the smallest R² drop and the smallest accuracy gain.

---
## Real-world example: where the two models actually disagree

The numbers above say `log1p` trades R² for typical-case accuracy, but they don't show *where* in the price range that trade happens. The two scatter plots below plot every validation listing: actual price on the x-axis, the model's predicted price on the y-axis. A perfect model would place every point exactly on the dashed diagonal.

Look at the dense cluster of points under $200 — the `log1p` model on the right hugs that diagonal noticeably more tightly than the raw model on the left. Now look at the sparse points above $600 — the raw model's guesses spread out more on the y-axis there, but at least they're spread around the right neighborhood. The `log1p` model pulls its guesses for the expensive listings in toward the middle of the price range, because on the log scale a $600 forecast for a $1,000 listing costs the model almost nothing.

In [ ]:
np.random.seed(42)

raw_pipe = Pipeline([('prep', _prep), ('lr', LinearRegression())]).fit(_Xtr, _ytr)
raw_pred_scatter = raw_pipe.predict(_Xva)

log_pipe = Pipeline([('prep', _prep), ('lr', LinearRegression())]).fit(_Xtr, np.log1p(_ytr))
log_pred_scatter = np.expm1(log_pipe.predict(_Xva))

r2_raw_s = r2_score(_yva, raw_pred_scatter)
med_raw_s = float(np.median(np.abs(_yva.values - raw_pred_scatter)))
r2_log_s = r2_score(_yva, log_pred_scatter)
med_log_s = float(np.median(np.abs(_yva.values - log_pred_scatter)))

fig = make_subplots(
    rows=1, cols=2, shared_yaxes=True,
    subplot_titles=(
        f"Raw price model — R²={r2_raw_s:.3f}, median error ${med_raw_s:.0f}",
        f"log1p(price) model — R²={r2_log_s:.3f}, median error ${med_log_s:.0f}",
    ),
)

_diag = [0, 1000]
for _col, _pred in [(1, raw_pred_scatter), (2, log_pred_scatter)]:
    fig.add_trace(go.Scatter(
        x=_yva, y=_pred, mode="markers",
        marker=dict(color=PALETTE["primary"], size=4, opacity=0.35),
        showlegend=False,
    ), row=1, col=_col)
    fig.add_trace(go.Scatter(
        x=_diag, y=_diag, mode="lines",
        line=dict(color=PALETTE["muted"], dash="dash", width=1.5),
        showlegend=False,
    ), row=1, col=_col)

fig.update_xaxes(title_text="Actual price ($)", range=[0, 1000], gridcolor="#DEE2E6")
fig.update_yaxes(title_text="Predicted price ($)", range=[0, 1000], gridcolor="#DEE2E6", col=1)
fig.update_layout(
    height=460,
    paper_bgcolor=PALETTE["background"], plot_bgcolor=PALETTE["surface"],
    font=dict(family=FONT["family"], size=FONT["size_tick"]),
)
fig.show()


> **Discussion question:** A host who owns three $90 studios and a host who owns one $2,000 penthouse both rely on this model's price guidance. Which model — raw price or `log1p` — would each host prefer, and why might a marketplace like Airbnb need to care about both hosts, not just one?

---
## Key takeaway

> **A skew-compressing transform on the target isn't automatically an improvement: it can lower R² while cutting the typical listing's error by 20%, because it trades accuracy on rare expensive listings for accuracy on the common cheap ones — which one you want depends on which listings your model actually needs to get right.**

---
*Next up: 07_feature_selection — where you learn to drop the features that hurt more than they help*